### Canibalización entre ítems de una vitrina

Se estima el parámetro de canibalización $\beta$ con los logs de la política random del Open Bandit Dataset. La idea es ver si un ítem pierde clics cuando comparte la vitrina con ítems de su misma categoría. El modelo que planteamos es

$$P(\text{clic en } k) = \theta_k(a_k) - \beta \cdot c_k,$$

donde $c_k$ es cuántos de los otros dos ítems de la vitrina son de la misma categoría que el ítem en la posición $k$. 

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

OBD_PATH = Path("Contenido/Datasets/open_bandit_dataset/open_bandit_dataset")
CAMPAIGN = "men"

In [2]:
UF = [f"user_feature_{i}" for i in range(4)]


def leer_log(politica, campana):
    log = pd.read_csv(OBD_PATH / politica / campana / f"{campana}.csv",
                      usecols=["timestamp", "item_id", "position", "click"] + UF,
                      dtype={c: "category" for c in UF})
    ts = log["timestamp"].str.replace(r"(\d{2}:\d{2}:\d{2})\+", r"\1.000000+", regex=True)
    log["timestamp"] = pd.to_datetime(ts, format="%Y-%m-%d %H:%M:%S.%f%z")
    return log.sort_values("timestamp", kind="stable").reset_index(drop=True)


log = leer_log("random", CAMPAIGN)
pd.DataFrame({"Impresiones": [len(log)], "Timestamps distintos": [log["timestamp"].nunique()],
              "Ítems": [log["item_id"].nunique()], "CTR": [log["click"].mean()]})

,Impresiones,Timestamps distintos,Ítems,CTR
0,452949,439004,34,0.005124


## Reconstrucción de vitrinas

El dataset tiene una fila por impresión y no trae un identificador de vitrina, así que para saber qué ítems se mostraron juntos se agrupan las tres filas de cada vitrina.

Las filas de una misma vitrina no tienen exactamente el mismo timestamp (casi todos son distintos), pero aparecen seguidas, con el mismo usuario y separadas por milisegundos. Entonces se parte una vitrina nueva cuando:
- pasan más de $\tau$ segundos desde la fila anterior,
- cambian las características del usuario, o
- se repite una posición.

In [3]:
def reconstruir_vitrinas(log, tau):
    gap = log["timestamp"].diff().dt.total_seconds().fillna(np.inf).to_numpy()
    mismo_usuario = (log[UF] == log[UF].shift()).all(axis=1).to_numpy()
    posiciones = log["position"].to_numpy()

    vitrina = np.empty(len(log), dtype=int)
    actual, vistas = -1, set()
    for i in range(len(log)):
        if gap[i] > tau or not mismo_usuario[i] or posiciones[i] in vistas:
            actual, vistas = actual + 1, set()
        vistas.add(posiciones[i])
        vitrina[i] = actual
    return vitrina


filas = []
for tau in (0.05, 0.1, 0.5):
    tamano = np.bincount(reconstruir_vitrinas(log, tau))
    filas.append({"tau (s)": tau, "Vitrinas": len(tamano),
                  "Con 1 fila": (tamano == 1).sum(), "Con 2 filas": (tamano == 2).sum(),
                  "Con 3 filas": (tamano == 3).sum(),
                  "Impresiones en vitrinas completas": np.mean(np.repeat(tamano, tamano) == 3)})
pd.DataFrame(filas)

,tau (s),Vitrinas,Con 1 fila,Con 2 filas,Con 3 filas,Impresiones en vitrinas completas
0,0.05,154748,4212,2871,147665,0.978024
1,0.10,153398,2633,1979,148786,0.985449
2,0.50,151976,1024,931,150021,0.993628


Con $\tau = 0{,}1$ el 98,5 % de las impresiones queda en vitrinas completas, y el resultado cambia poco con 0,05 o 0,5. Por lo tanto, nos quedamos con $\tau = 0{,}1$ y descartamos las vitrinas incompletas.

In [4]:
log["vitrina"] = reconstruir_vitrinas(log, 0.1)
completas = log[log.groupby("vitrina")["position"].transform("size") == 3].sort_values(["vitrina", "position"])

# una fila por vitrina, con el ítem y el clic de las posiciones 1, 2 y 3
vitrinas = completas["item_id"].to_numpy().reshape(-1, 3)
clics = completas["click"].to_numpy().reshape(-1, 3)
pd.DataFrame(vitrinas[:5], columns=["Posición 1", "Posición 2", "Posición 3"])

,Posición 1,Posición 2,Posición 3
0,0,23,25
1,25,30,17
2,25,30,17
3,25,30,17
4,0,23,25


Antes de estimar, se revisa cómo arma las vitrinas la política random. Cada ítem aparece en cada posición con la misma probabilidad (1/34 en *men*), pero eso no significa que las vitrinas se sorteen de forma uniforme. Para revisarlo, se compara cuántos órdenes distintos aparecen para cada conjunto de tres ítems con lo que se esperaría si el orden fuera al azar.

In [5]:
conjunto = pd.Series([tuple(sorted(v)) for v in vitrinas])
orden = pd.Series([tuple(v) for v in vitrinas])
n_por_conjunto = conjunto.value_counts()
m = log["item_id"].nunique()

pd.Series({
    "Conjuntos de 3 ítems observados": conjunto.nunique(),
    "Conjuntos posibles": m * (m - 1) * (m - 2) // 6,
    "Vitrinas ordenadas distintas": orden.nunique(),
    "Vitrinas ordenadas posibles": m * (m - 1) * (m - 2),
    "Órdenes distintos por conjunto (observado)": orden.groupby(conjunto).nunique().mean(),
    # con n apariciones al azar, cada uno de los 6 órdenes aparece con prob. 1 - (5/6)^n
    "Órdenes distintos por conjunto (si fuera uniforme)": (6 * (1 - (5 / 6) ** n_por_conjunto)).mean(),
}).round(2)

Conjuntos de 3 ítems observados                        5846.00
Conjuntos posibles                                     5984.00
Vitrinas ordenadas distintas                          16813.00
Vitrinas ordenadas posibles                           35904.00
Órdenes distintos por conjunto (observado)                2.88
Órdenes distintos por conjunto (si fuera uniforme)        5.63
dtype: float64

Casi todos los conjuntos de ítems aparecen, pero cada uno se muestra en promedio en 2,9 de sus 6 órdenes posibles, cuando con un orden uniforme serían 5,6. Además, la misma vitrina suele repetirse en impresiones seguidas. Es decir, random no sortea vitrinas de forma uniforme. Esto no impide estimar $\beta$, porque la vitrina se sigue eligiendo sin mirar al usuario, pero como las mismas vitrinas se repiten, los errores estándar se agrupan por conjunto de ítems y no por vitrina.

## Vecinos de la misma categoría

Usamos como categoría de cada ítem la columna item_feature_1 del archivo item_context.csv. Para cada posición contamos cuántos de los otros dos ítems de la vitrina son de su misma categoría.

In [6]:
def contar_vecinos(cat):

    # paso a int antes de sumar 
    igual_12 = (cat[:, 0] == cat[:, 1]).astype(int)
    igual_13 = (cat[:, 0] == cat[:, 2]).astype(int)
    igual_23 = (cat[:, 1] == cat[:, 2]).astype(int)
    return np.column_stack([igual_12 + igual_13, igual_12 + igual_23, igual_13 + igual_23])


item_context = pd.read_csv(OBD_PATH / "random" / CAMPAIGN / "item_context.csv", index_col=0).sort_values("item_id")
categoria = item_context["item_feature_1"].astype("category").cat.codes.to_numpy()
n_items = len(categoria)

vecinos = contar_vecinos(categoria[vitrinas])

pd.DataFrame({"Vecinos de la misma categoría": vecinos.ravel(), "click": clics.ravel()}) \
  .groupby("Vecinos de la misma categoría")["click"].agg(Impresiones="size", CTR="mean")

,Impresiones,CTR
Vecinos de la misma categoría,,
0,329195,0.004970
1,109036,0.004760
2,8127,0.004184


## Estimación de $\beta$

Estimamos una regresión lineal del clic sobre el número de vecinos de la misma categoría, con un efecto fijo por cada combinación ítem × posición.

$$r_{ik} = \alpha_{a_{ik},k} - \beta \cdot c_{ik} + \varepsilon_{ik}.$$

Los efectos fijos capturan qué tan atractivo es cada ítem en cada posición, así que $\beta$ sale de comparar un mismo ítem, en una misma posición, con distintos vecinos. Como en random los vecinos no dependen del usuario, esa comparación es causal.

Para estimar con efectos fijos se le resto a cada variable su promedio dentro de su celda ítem × posición y después hago una regresión simple. 

In [7]:
def estimar_beta(vitrinas, clics, vecinos, grupos, n_items):
    item, pos = vitrinas.ravel(), np.tile(np.arange(3), len(vitrinas))
    r, c = clics.ravel().astype(float), vecinos.ravel().astype(float)

    # le resto a r y a c su promedio 
    celda = item * 3 + pos
    n_celda = np.bincount(celda, minlength=n_items * 3)
    r_dm = r - (np.bincount(celda, r, n_items * 3) / n_celda)[celda]
    c_dm = c - (np.bincount(celda, c, n_items * 3) / n_celda)[celda]
    pendiente = (c_dm * r_dm).sum() / (c_dm ** 2).sum()

    # error estándar por grupos
    residuo = r_dm - pendiente * c_dm
    aporte_grupo = np.bincount(np.repeat(grupos, 3), c_dm * residuo)
    se = np.sqrt((aporte_grupo ** 2).sum()) / (c_dm ** 2).sum()
    return -pendiente, se   # beta es la caída, por eso cambio el signo


# agrupamos los errores por conjunto de ítems
conjunto = pd.Series([tuple(sorted(v)) for v in vitrinas])
grupos = conjunto.astype("category").cat.codes.to_numpy()
beta, se = estimar_beta(vitrinas, clics, vecinos, grupos, n_items)
ctr = clics.mean()

pd.Series({
    "beta": beta,
    "Error estándar": se,
    "IC 95% inferior": beta - 1.96 * se,
    "IC 95% superior": beta + 1.96 * se,
    "z": beta / se,
    "beta / CTR": beta / ctr,
    "Efecto mínimo detectable (80% potencia)": 2.8 * se,
}).round(5)

beta                                       0.00039
Error estándar                             0.00031
IC 95% inferior                           -0.00022
IC 95% superior                            0.00099
z                                          1.24430
beta / CTR                                 0.07866
Efecto mínimo detectable (80% potencia)    0.00087
dtype: float64

Por último, se revisa que el resultado no dependa de la ventana $\tau$. Para esto, y para lo que viene después, se juntan los pasos anteriores en dos funciones: una que, dado un log y su item_context, reconstruye las vitrinas y cuenta los vecinos, y otra que además estima $\beta$.

In [8]:
def vitrinas_campana(log, item_context, tau=0.1):
    categoria = item_context["item_feature_1"].astype("category").cat.codes.to_numpy()
    v = log.assign(vitrina=reconstruir_vitrinas(log, tau))
    v = v[v.groupby("vitrina")["position"].transform("size") == 3].sort_values(["vitrina", "position"])
    vit = v["item_id"].to_numpy().reshape(-1, 3)
    cl = v["click"].to_numpy().reshape(-1, 3)
    vec = contar_vecinos(categoria[vit])
    grp = pd.Series([tuple(sorted(x)) for x in vit]).astype("category").cat.codes.to_numpy()
    return vit, cl, vec, grp, categoria


def beta_campana(log, item_context, tau=0.1):
    vit, cl, vec, grp, categoria = vitrinas_campana(log, item_context, tau)
    b, s = estimar_beta(vit, cl, vec, grp, len(categoria))
    return {"Vitrinas": len(vit), "CTR": cl.mean(), "beta": b, "Error estándar": s}


pd.DataFrame([{"tau (s)": tau, **beta_campana(log, item_context, tau)} for tau in (0.05, 0.1, 0.5)]).round(6)

,tau (s),Vitrinas,CTR,beta,Error estándar
0,0.05,147665,0.004892,0.000363,0.000309
1,0.10,148786,0.004904,0.000386,0.000310
2,0.50,150021,0.004895,0.000382,0.000309


## Resumen de $\beta$

Para las secciones siguientes se guardan el log de *men* y el resultado de $\beta$ con $\tau = 0{,}1$.

In [9]:
logs = {CAMPAIGN: (log, item_context)}   
res = beta_campana(log, item_context)

comparacion = pd.DataFrame([{"Campaña": CAMPAIGN, "Ítems": len(item_context),
                             "Categorías": item_context["item_feature_1"].nunique(),
                             "Impresiones": len(log), **res}]).set_index("Campaña")
comparacion["IC 95% inferior"] = comparacion["beta"] - 1.96 * comparacion["Error estándar"]
comparacion["IC 95% superior"] = comparacion["beta"] + 1.96 * comparacion["Error estándar"]
comparacion["beta / CTR"] = comparacion["beta"] / comparacion["CTR"]
comparacion["Efecto mínimo detectable"] = 2.8 * comparacion["Error estándar"]
comparacion.round(5)

,Ítems,Categorías,Impresiones,Vitrinas,CTR,beta,Error estándar,IC 95% inferior,IC 95% superior,beta / CTR,Efecto mínimo detectable
Campaña,,,,,,,,,,,
men,34,7,452949,148786,0.0049,0.00039,0.00031,-0.00022,0.00099,0.07866,0.00087


## Sesgo por canibalización al evaluar bts con logs de random

En el benchmark de Saito et al. se estima el CTR de la política bts usando los logs de random y se compara con el CTR observado de bts. El IPW por posición corrige qué ítem aparece en cada posición, pero en los logs de random cada ítem aparece con los vecinos que eligió random, no con los que elegiría bts. Si bts junta más (o menos) ítems de la misma categoría que random, el IPW por posición tiene un sesgo $\beta\Delta$, con

$$\Delta = \sum_k \sum_a \pi_{\text{bts}}(a \mid k)\Big(\mathbb{E}_{\text{bts}}[c_k \mid a_k = a] - \mathbb{E}_{\text{random}}[c_k \mid a_k = a]\Big).$$

Es decir, para cada ítem en cada posición comparo cuántos vecinos de su categoría tuvo en bts y en random, ponderando por qué tan seguido lo muestra bts.

In [10]:
def vecinos_por_item(vitrinas, vecinos, n_items):
    veces = np.column_stack([np.bincount(vitrinas[:, k], minlength=n_items) for k in range(3)])
    suma = np.column_stack([np.bincount(vitrinas[:, k], vecinos[:, k], n_items) for k in range(3)])
    return veces, suma / np.maximum(veces, 1)


filas = []
for campana, (log_r, ic_c) in logs.items():
    log_b = leer_log("bts", campana)
    vit_r, _, vec_r, _, categoria = vitrinas_campana(log_r, ic_c)
    vit_b, _, vec_b, _, _ = vitrinas_campana(log_b, ic_c)
    n_items = len(categoria)

    _, vecinos_random = vecinos_por_item(vit_r, vec_r, n_items)
    veces_bts, vecinos_bts = vecinos_por_item(vit_b, vec_b, n_items)
    pi_bts = veces_bts / veces_bts.sum(axis=0)          # pi_bts(a | k)
    pi_random = np.full((n_items, 3), 1 / n_items)       # pi_random(a | k), uniforme como en obp

    b, s = comparacion.loc[campana, "beta"], comparacion.loc[campana, "Error estándar"]
    direcciones = {
        # política evaluada: (pi_e, vecinos con pi_e, vecinos en los logs usados, CTR real de la evaluada)
        "bts": (pi_bts, vecinos_bts, vecinos_random, log_b["click"].mean()),
        "random": (pi_random, vecinos_random, vecinos_bts, log_r["click"].mean()),
    }
    for evaluada, (pi_e, vecinos_e, vecinos_logs, ctr_real) in direcciones.items():
        con_pi_e = (pi_e * vecinos_e).sum()          # vecinos parecidos por vitrina con la política evaluada
        en_logs = (pi_e * vecinos_logs).sum()        # lo que asume el IPW por posición al usar los logs de la otra
        delta = con_pi_e - en_logs
        sesgos = sorted([(b - 1.96 * s) * delta / 3, (b + 1.96 * s) * delta / 3])
        filas.append({
            "Campaña": campana,
            "Política evaluada": evaluada,
            "Logs usados": "random" if evaluada == "bts" else "bts",
            "Vecinos parecidos (política evaluada)": con_pi_e,
            "Vecinos parecidos (logs usados)": en_logs,
            "Delta": delta,
            "CTR real": ctr_real,
            "Sesgo (CTR por impresión)": b * delta / 3,
            "IC 95% inferior": sesgos[0],
            "IC 95% superior": sesgos[1],
            "Sesgo / CTR real": b * delta / 3 / ctr_real,
        })
    del log_b   # el log de bts es grande, lo libero

sesgo_canibalizacion = pd.DataFrame(filas).set_index(["Campaña", "Política evaluada"])
sesgo_canibalizacion

Logs usados  ...  Sesgo / CTR real
Campaña Política evaluada              ...                  
men     bts                    random  ...          0.001631
        random                    bts  ...         -0.001011

[2 rows x 9 columns]

## Descomposición del error de los estimadores

En ope.ipynb se estima el CTR de random (campaña *men*) con 200 muestras, usando siete estimadores. Con esas 200 estimaciones se puede separar el error de cada estimador. Primero, el error cuadrático medio se divide en sesgo y varianza:

$$\text{MSE} = \underbrace{\big(\overline{V} - V_{\text{real}}\big)^2}_{\text{sesgo}^2} + \underbrace{\frac{1}{200}\sum_{s}\big(\hat V_s - \overline{V}\big)^2}_{\text{varianza}},$$

donde $\hat V_s$ es la estimación de la semilla $s$, $\overline{V}$ su promedio y $V_{\text{real}}$ el CTR observado de random. Luego, el sesgo se divide en la parte que viene de la canibalización, $\beta\Delta/3$, y el resto, que viene de otras fuentes, como puede ser un modelo de clics mal especificado, errores en los propensity scores de *bts* o cambios entre los periodos de las dos políticas.

El CTR real también tiene ruido, porque es un promedio de 452.949 impresiones. Por eso se reporta el error estándar del sesgo total, que combina la variación entre semillas y el error estándar del CTR real.

In [11]:
ope = pd.read_csv("resultados/ope_men_n10000_tuning_por_semilla.csv")   

log_r = logs["men"][0]
ctr_real = log_r["click"].mean()
se_ctr_real = np.sqrt(ctr_real * (1 - ctr_real) / len(log_r))
canib = sesgo_canibalizacion.loc[("men", "random")]

por_estimador = ope.groupby("estimador")["estimado"]
n_semillas = por_estimador.size()
media = por_estimador.mean()
sesgo = media - ctr_real
varianza = por_estimador.var(ddof=0)
mse = sesgo ** 2 + varianza                    
se_sesgo = np.sqrt(varianza / n_semillas + se_ctr_real ** 2)

descomposicion = pd.DataFrame({
    "CTR estimado (promedio)": media,
    "Sesgo total": sesgo,
    "Error estándar del sesgo": se_sesgo,
    "Varianza": varianza,
    "RMSE x 10^3": np.sqrt(mse) * 1000,
    "% del MSE por sesgo": sesgo ** 2 / mse,
    "% del MSE por varianza": varianza / mse,
    "Sesgo por canibalización": canib["Sesgo (CTR por impresión)"],
    "Sesgo de otras fuentes": sesgo - canib["Sesgo (CTR por impresión)"],
    "Canibalización / sesgo total": canib["Sesgo (CTR por impresión)"] / sesgo,
}).sort_values("RMSE x 10^3")

print(f"CTR real de random: {ctr_real:.5f} (error estándar {se_ctr_real:.5f})")
print(f"Sesgo por canibalización: {canib['Sesgo (CTR por impresión)']:+.2e} "
      f"(IC 95% [{canib['IC 95% inferior']:+.2e}, {canib['IC 95% superior']:+.2e}])")
for col in ["Sesgo total", "Error estándar del sesgo", "Varianza", "Sesgo por canibalización", "Sesgo de otras fuentes"]:
    descomposicion[col] = descomposicion[col].map("{:.2e}".format)
descomposicion.round(4)

CTR real de random: 0.00512 (error estándar 0.00011)
Sesgo por canibalización: -5.18e-06 (IC 95% [-1.33e-05, +2.98e-06])


,CTR estimado (promedio),Sesgo total,Error estándar del sesgo,Varianza,RMSE x 10^3,% del MSE por sesgo,% del MSE por varianza,Sesgo por canibalización,Sesgo de otras fuentes,Canibalización / sesgo total
estimador,,,,,,,,,,
DM,0.0064,1.25e-03,1.23e-04,7.76e-07,1.5308,0.6690,0.3310,-5.18e-06,1.26e-03,-0.0041
DRos,0.0058,6.59e-04,1.75e-04,3.85e-06,2.0707,0.1012,0.8988,-5.18e-06,6.64e-04,-0.0079
Switch-DR,0.0057,5.28e-04,2.09e-04,6.47e-06,2.5976,0.0412,0.9588,-5.18e-06,5.33e-04,-0.0098
IPW,0.0056,4.41e-04,2.10e-04,6.58e-06,2.6023,0.0287,0.9713,-5.18e-06,4.46e-04,-0.0118
SNIPW,0.0055,3.95e-04,2.15e-04,7.00e-06,2.6757,0.0218,0.9782,-5.18e-06,4.00e-04,-0.0131
SNDR,0.0056,4.49e-04,2.15e-04,6.98e-06,2.6801,0.0281,0.9719,-5.18e-06,4.55e-04,-0.0115
DR,0.0052,1.03e-04,3.03e-04,1.61e-05,4.0147,0.0007,0.9993,-5.18e-06,1.08e-04,-0.0504


**Declaración de uso de IA**

Utilicé IA para que se entienda de forma correcta lo que hace cada parte del código y la construcción teórica matemática de cada segmento (markdowns) y para tratar inicialmente los datos (recuperar la información relevante al momento de cargar los datos). Además, se utilizó como apoyo en el código, para arreglar errores y hacer ajustes.